In [1]:
import jax
import jax.numpy as jnp
from flax import linen as nn
from jax import random
from flax.linen.initializers import normal

jax.config.update("jax_enable_x64", True)
jax.config.update('jax_default_matmul_precision', 'highest')


def compute_jacobian(params, model, x):
    jacobian_fn = jax.jacrev(model.apply, argnums=0)
    return jacobian_fn(params, x)


# Might be able to use ravel eventually but this is easier
def flatten_jacobian(jacobian_dict, x):
    batch_size = x.shape[0]
    flat_jacobian = []
    for layer in jacobian_dict['params'].values():
        for param in layer.values():
            flat_jacobian.append(param.reshape(batch_size, -1))
    return jnp.concatenate(flat_jacobian, axis=1)


# Might be able to use ravel eventually but this is easier
def flatten_grad(grad_dict, x):
    """

    :param grad_dict:
    :param x: Sample batch data
    :return:
    """
    batch_size = x.shape[0]
    flat_grad = []
    for layer in grad_dict['params'].values():
        for param in layer.values():
            flat_grad.append(param.reshape(-1))
    return jnp.concatenate(flat_grad)


In [6]:
# Define a simple neural network using Flax
class SimpleNN(nn.Module):
    hidden_dim: int = 100

    @nn.compact
    def __call__(self, x):
        x = nn.Dense(features=self.hidden_dim, use_bias=True, param_dtype=jnp.float64,
                     kernel_init=normal(stddev=1),
                     bias_init=normal(stddev=1))(x)
        x = nn.relu(x)
        x = nn.Dense(1, use_bias=False, param_dtype=jnp.float64)(x)
        return x


In [10]:
# Initialize the model and parameters
key1, key2 = random.split(random.PRNGKey(2))
model = SimpleNN(hidden_dim=5)
x = jnp.ones([1], dtype=jnp.float64)
params = model.init(key2, x)

# print(model.tabulate(
#     jax.random.key(0),  jnp.ones([10, 1]), compute_flops=True,
#     compute_vjp_flops=True)
# )

x = random.normal(key1, (10,1), dtype=jnp.float64)  # Example input
J_0 = compute_jacobian(params, model, x)
J_0 = flatten_jacobian(J_0, x)
print(jnp.linalg.matrix_rank(J_0))
sigma_min = jnp.sort(jnp.linalg.eigvalsh(J_0 @ J_0.T))
print(sigma_min)


8
[-2.09450721e-16  8.04437760e-16  4.21790057e-04  7.98176767e-04
  3.51326562e-03  3.92906249e-02  2.74096496e-01  8.65862379e-01
  4.80121561e+00  6.24080316e+01]
